# ETF_V11 2021 Validation Framework

## 目的

验证 ETF 机器学习轮动策略在 2021 年之后的 ETF 生态中是否具备可用性、稳定性和鲁棒性。

本 notebook 以 V10 为基础，但只保留固定 2021 起点训练窗口，并新增：

- `top1 / top3 / top5`
- 成本按万分之一，即 `COST_RATE = 0.0001`
- 换手与成本后收益
- 多 seed random topN
- 简单 `ret_20` / `trend_score_25` baseline
- 共同 OOS 比较
- 训练窗口间持仓重合率和收益相关性

第一阶段目标是验证策略是否可信，不做新特征和模型调参。

In [ ]:
# =========================
# 0. Config
# =========================
try:
    from jqdata import *
except ImportError:
    print("jqdata is unavailable; cached-panel parts can still run if no JoinQuant API call is needed.")
import os
import gc
import pickle
import shutil
import datetime
import math
import numpy as np
import pandas as pd

try:
    from tqdm import tqdm
except Exception:
    def tqdm(iterable, **kwargs):
        desc = kwargs.get("desc", "progress")
        try:
            total = len(iterable)
        except Exception:
            total = None
        if total is not None:
            print(desc + ": total=" + str(total))
        else:
            print(desc)
        return iterable

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


def unique_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


def drop_duplicate_columns(df):
    if df is None or df.empty:
        return df
    if df.columns.is_unique:
        return df
    dup_cols = [c for c in df.columns[df.columns.duplicated()].tolist()]
    print("drop duplicate columns:", dup_cols[:20], "count=", len(dup_cols))
    return df.loc[:, ~df.columns.duplicated()].copy()

OUT_DIR = "etf_ml_v11_2021_validation_outputs"
MODEL_DIR = os.path.join(OUT_DIR, "models")
CANDIDATE_MODEL_DIR = os.path.join(OUT_DIR, "candidate_models")
PANEL_CSV = os.path.join(OUT_DIR, "etf_ml_v11_weekly_panel.csv")
SCORE_CSV = os.path.join(OUT_DIR, "etf_ml_v11_score_panel.csv")
MODEL_MANIFEST_CSV = os.path.join(OUT_DIR, "etf_ml_v11_model_manifest.csv")
SUMMARY_CSV = os.path.join(OUT_DIR, "etf_ml_v11_summary.csv")
YEARLY_CSV = os.path.join(OUT_DIR, "etf_ml_v11_yearly.csv")
WEEKLY_PROXY_CSV = os.path.join(OUT_DIR, "etf_ml_v11_weekly_proxy.csv")
LATEST_TARGETS_CSV = os.path.join(OUT_DIR, "etf_ml_v11_latest_targets.csv")
MONEY_THRESHOLD_SUMMARY_CSV = os.path.join(OUT_DIR, "etf_ml_v11_money_threshold_summary.csv")
CANDIDATE_MODELS_CSV = os.path.join(OUT_DIR, "etf_ml_v11_candidate_models.csv")
TARGET_OVERLAP_CSV = os.path.join(OUT_DIR, "etf_ml_v11_target_overlap.csv")
SCORE_GAP_DIAGNOSTICS_CSV = os.path.join(OUT_DIR, "etf_ml_v11_score_gap_diagnostics.csv")
DRAWDOWN_EVENTS_CSV = os.path.join(OUT_DIR, "etf_ml_v11_drawdown_events.csv")

SOURCE_PANEL_CANDIDATES = [
    PANEL_CSV,
    "etf_ml_v2_outputs/etf_ml_v2_weekly_panel.csv",
    "../etf_ml_v2_outputs/etf_ml_v2_weekly_panel.csv",
    "etf_ml_v8_v2_jq_like_robustness_outputs/etf_ml_v8_weekly_panel.csv",
]

# AUTO: load existing panel if possible; otherwise rebuild in JoinQuant research environment.
REBUILD_DATA = "AUTO"  # True / False / "AUTO"
START_DATE = "2021-01-01"
END_DATE = "2026-06-30"
LOOKBACK_DAYS = 60
LABEL_HORIZONS = [5]
MIN_LISTING_DAYS = 180
MIN_AVG_MONEY_20 = 50000000.0  # V11 fixes the liquidity floor to avoid low-liquidity ETF noise.
MIN_AVG_MONEY_20_GRID = [50000000.0]
ETF_CHUNK_SIZE = 120
BENCHMARK = "000985.XSHG"
RANDOM_SEED = 42

# V2 features. Keep these fixed for anchor-grid experiments.
TREND_WINDOWS = [10, 20, 25, 60]
TREND_WEIGHT_END = 2.0
PRICE_HISTORY_COUNT = max(LOOKBACK_DAYS + 1, max(TREND_WINDOWS) + 1)
POOL_CONTEXT_WINDOW = 25

EXCLUDE_NAME_KEYWORDS = [
    "债", "国债", "地债", "政金债", "公司债", "城投", "可转债",
    "货币", "现金", "快线", "快钱", "同业存单",
]

ETF_GROUP_RULES = [
    ("bank", ["银行"]),
    ("innovative_drug", ["创新药", "新药", "医药", "医疗", "生物", "药"]),
    ("oil_gas", ["油气", "石油", "能源"]),
    ("coal", ["煤炭"]),
    ("semiconductor", ["半导", "芯片", "集成电路"]),
    ("software_ai", ["软件", "人工智能", "AI", "云计算", "大数据", "计算机", "信创"]),
    ("broker", ["证券", "券商"]),
    ("military", ["军工", "国防", "航空航天", "通用航空"]),
    ("gold", ["黄金"]),
    ("nonferrous", ["有色", "稀有金属", "稀土"]),
    ("consumer", ["消费", "食品", "酒"]),
    ("real_estate", ["地产", "房地产"]),
    ("new_energy", ["新能源", "光伏", "电池", "锂电", "储能"]),
]

TRAIN_WINDOWS = [
    # Fixed 2021 start. ETF universe before 2021 is intentionally excluded.
    ("train20210101_20231231", "2021-01-01", "2023-12-31"),
    ("train20210101_20241231", "2021-01-01", "2024-12-31"),
    ("train20210101_20251231", "2021-01-01", "2025-12-31"),
]

TARGET_SPECS = [
    # Keep this experiment single-label. Feature and model changes belong to V11.
    ("ret5d", "future_ret_5d", "future_ret_5d"),
]

TOP_N_LIST = [1, 3, 5]
GROUP_CAP_LIST = [0]  # No theme cap for V11 validation.
COST_RATE = 0.0001  # 万分之一 = 0.01% = 1 bp, applied to traded notional.
CANDIDATE_MAX_OVERALL = 10
CANDIDATE_MAX_PER_MONEY = 10

# Keep label horizon and portfolio rebalance cadence aligned.
# 20 trading days is treated as a monthly model in exported bundles/backtests.
TARGET_REBALANCE_RULES = {
    "ret3d": {"rebalance_mode": "weekly", "rebalance_interval_weeks": 1},
    "ret5d": {"rebalance_mode": "weekly", "rebalance_interval_weeks": 1},
    "ret10d": {"rebalance_mode": "weekly", "rebalance_interval_weeks": 2},
    "ret20d": {"rebalance_mode": "monthly", "rebalance_interval_weeks": 4},
    "alpha5d": {"rebalance_mode": "weekly", "rebalance_interval_weeks": 1},
    "alpha10d": {"rebalance_mode": "weekly", "rebalance_interval_weeks": 2},
    "alpha20d": {"rebalance_mode": "monthly", "rebalance_interval_weeks": 4},
}

BASE_PRICE_FEATURE_COLS = [
    "ret_1", "ret_5", "ret_10", "ret_20", "ret_60",
    "vol_5", "vol_20", "vol_60",
    "close_to_ma20", "close_to_ma60", "ma5_to_ma20", "ma20_to_ma60",
    "drawdown_20", "drawdown_60",
    "amp_20", "amp_60",
    "money_mean_20", "money_ratio_5_20", "money_ratio_20_60",
    "volume_ratio_5_20", "volume_ratio_20_60",
    "max_ret_20", "min_ret_20",
]
TREND_FEATURE_COLS = []
for _w in TREND_WINDOWS:
    TREND_FEATURE_COLS.extend([
        "trend_ann_%s" % _w,
        "trend_r2_%s" % _w,
        "trend_score_%s" % _w,
        "trend_vol_%s" % _w,
        "trend_score_vol_adj_%s" % _w,
        "trend_simple_ann_%s" % _w,
    ])
CONTEXT_FEATURE_COLS = [
    "pool_breadth_%s" % POOL_CONTEXT_WINDOW,
    "pool_median_vol_%s" % POOL_CONTEXT_WINDOW,
]
RAW_FEATURE_COLS = BASE_PRICE_FEATURE_COLS + TREND_FEATURE_COLS
RANK_FEATURE_COLS = ["rank_" + c for c in RAW_FEATURE_COLS]
FEATURE_COLS = RAW_FEATURE_COLS + RANK_FEATURE_COLS + CONTEXT_FEATURE_COLS

LGB_PARAMS = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.04,
    "num_leaves": 31,
    "max_depth": 5,
    "min_data_in_leaf": 80,
    "feature_fraction": 0.90,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 1.0,
    "min_gain_to_split": 0.0,
    "verbose": -1,
    "seed": RANDOM_SEED,
}
NUM_BOOST_ROUND = 180

for _p in [OUT_DIR, MODEL_DIR, CANDIDATE_MODEL_DIR]:
    if not os.path.exists(_p):
        os.makedirs(_p)

print("out dir:", OUT_DIR)
print("features:", len(FEATURE_COLS))
print("train windows:", TRAIN_WINDOWS)
print("target specs:", TARGET_SPECS)
print("money thresholds:", MIN_AVG_MONEY_20_GRID)
print("V11 focus: 2021-start validation with cost, topN, common OOS, and stability checks.")

In [ ]:
# =========================
# 1. Feature and data helpers
# =========================
ETF_NAME_CACHE = {}


def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def safe_ratio(a, b):
    if pd.isnull(a) or pd.isnull(b) or b == 0:
        return np.nan
    return float(a) / float(b)


def safe_ret(close, days):
    if len(close) <= days:
        return np.nan
    base = close.iloc[-days - 1]
    if pd.isnull(base) or base <= 0:
        return np.nan
    return close.iloc[-1] / base - 1.0


def get_weekly_feature_dates(start_date, end_date):
    try:
        days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    except NameError:
        raise RuntimeError("Need JoinQuant API to rebuild ETF weekly panel. Set REBUILD_DATA=False and provide a cached panel if running outside JoinQuant.")
    if len(days) == 0:
        return []
    s = pd.Series(days)
    out = []
    for _, gdf in s.groupby(s.dt.strftime("%Y-%W")):
        out.append(pd.Timestamp(gdf.max()).normalize())
    return out


def should_exclude_name(name):
    text = str(name)
    for kw in EXCLUDE_NAME_KEYWORDS:
        if kw and kw in text:
            return True
    return False


def classify_etf_group(code, name):
    text = str(name)
    for group_name, keywords in ETF_GROUP_RULES:
        for kw in keywords:
            if kw and kw in text:
                return group_name
    return "single_" + str(code)


def get_etf_universe_on_date(date):
    try:
        sec_df = get_all_securities(["etf"], date=date)
    except NameError:
        raise RuntimeError("Need JoinQuant API to rebuild ETF universe. Use cached panel or run in JoinQuant.")
    except Exception as err:
        print("get_all_securities failed", date, err)
        return []
    if sec_df is None or sec_df.empty:
        return []
    out = []
    name_map = {}
    for code, row in sec_df.iterrows():
        try:
            start_date = row.get("start_date", None)
            if pd.isnull(start_date):
                start_date = get_security_info(code).start_date
            start_date = pd.Timestamp(start_date).date()
            feature_dt = pd.Timestamp(date).date()
            if feature_dt - start_date < datetime.timedelta(days=MIN_LISTING_DAYS):
                continue
            name = row.get("display_name", "")
            if should_exclude_name(name):
                continue
            out.append(code)
            name_map[code] = str(name)
        except Exception:
            continue
    ETF_NAME_CACHE[str(pd.Timestamp(date).date())] = name_map
    return out


def calc_trend_metrics(close, days):
    empty = {"ann": np.nan, "r2": np.nan, "score": np.nan, "vol": np.nan, "score_vol_adj": np.nan, "simple_ann": np.nan}
    if len(close) <= days:
        return empty
    recent = pd.Series(close.iloc[-(days + 1):].astype(float).values)
    if recent.isnull().any() or (recent <= 0).any():
        return empty
    y = np.log(recent.values)
    x = np.arange(len(y))
    weights = np.linspace(1.0, TREND_WEIGHT_END, len(y))
    try:
        slope, intercept = np.polyfit(x, y, 1, w=weights)
    except Exception:
        return empty
    ann = np.exp(slope * 250.0) - 1.0
    fit = slope * x + intercept
    ss_res = np.sum(weights * (y - fit) ** 2)
    ss_tot = np.sum(weights * (y - np.mean(y)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else 0.0
    r2 = max(0.0, min(1.0, float(r2)))
    daily_returns = recent.pct_change().dropna()
    vol = float(daily_returns.std() * np.sqrt(250.0)) if len(daily_returns) > 1 else np.nan
    period_ret = recent.iloc[-1] / recent.iloc[0] - 1.0
    simple_ann = (1.0 + period_ret) ** (250.0 / float(days)) - 1.0 if 1.0 + period_ret > 0 else np.nan
    score = ann * r2
    score_vol_adj = score * 0.20 / max(vol, 0.05) if not pd.isnull(vol) else np.nan
    return {"ann": ann, "r2": r2, "score": score, "vol": vol, "score_vol_adj": score_vol_adj, "simple_ann": simple_ann}


def calc_one_etf_features(code, price_df):
    df = price_df.sort_values("time").copy()
    if len(df) < max(40, int(LOOKBACK_DAYS * 0.8)):
        return None
    for col in ["open", "high", "low", "close", "volume", "money"]:
        if col not in df.columns:
            return None
    close = pd.Series(df["close"].astype(float).values)
    high = pd.Series(df["high"].astype(float).values)
    low = pd.Series(df["low"].astype(float).values)
    volume = pd.Series(df["volume"].astype(float).values)
    money = pd.Series(df["money"].astype(float).values)
    if close.isnull().any() or close.iloc[-1] <= 0:
        return None
    ret = close.pct_change()
    rec = {"code": code}
    rec["ret_1"] = safe_ret(close, 1)
    rec["ret_5"] = safe_ret(close, 5)
    rec["ret_10"] = safe_ret(close, 10)
    rec["ret_20"] = safe_ret(close, 20)
    rec["ret_60"] = safe_ret(close, 60)
    rec["vol_5"] = ret.tail(5).std()
    rec["vol_20"] = ret.tail(20).std()
    rec["vol_60"] = ret.tail(60).std()
    rec["close_to_ma20"] = safe_ratio(close.iloc[-1], close.tail(20).mean()) - 1
    rec["close_to_ma60"] = safe_ratio(close.iloc[-1], close.tail(60).mean()) - 1
    rec["ma5_to_ma20"] = safe_ratio(close.tail(5).mean(), close.tail(20).mean()) - 1
    rec["ma20_to_ma60"] = safe_ratio(close.tail(20).mean(), close.tail(60).mean()) - 1
    rec["drawdown_20"] = safe_ratio(close.iloc[-1], close.tail(20).max()) - 1
    rec["drawdown_60"] = safe_ratio(close.iloc[-1], close.tail(60).max()) - 1
    rec["amp_20"] = (high.tail(20) / low.tail(20) - 1).replace([np.inf, -np.inf], np.nan).mean()
    rec["amp_60"] = (high.tail(60) / low.tail(60) - 1).replace([np.inf, -np.inf], np.nan).mean()
    rec["money_mean_20"] = money.tail(20).mean()
    rec["money_ratio_5_20"] = safe_ratio(money.tail(5).mean(), money.tail(20).mean()) - 1
    rec["money_ratio_20_60"] = safe_ratio(money.tail(20).mean(), money.tail(60).mean()) - 1
    rec["volume_ratio_5_20"] = safe_ratio(volume.tail(5).mean(), volume.tail(20).mean()) - 1
    rec["volume_ratio_20_60"] = safe_ratio(volume.tail(20).mean(), volume.tail(60).mean()) - 1
    rec["max_ret_20"] = ret.tail(20).max()
    rec["min_ret_20"] = ret.tail(20).min()
    for w in TREND_WINDOWS:
        tm = calc_trend_metrics(close, int(w))
        rec["trend_ann_%s" % w] = tm["ann"]
        rec["trend_r2_%s" % w] = tm["r2"]
        rec["trend_score_%s" % w] = tm["score"]
        rec["trend_vol_%s" % w] = tm["vol"]
        rec["trend_score_vol_adj_%s" % w] = tm["score_vol_adj"]
        rec["trend_simple_ann_%s" % w] = tm["simple_ann"]
    return rec


def add_pool_context_features(df):
    out = df.copy()
    if len(out) == 0:
        return out
    ann_col = "trend_ann_%s" % POOL_CONTEXT_WINDOW
    vol_col = "trend_vol_%s" % POOL_CONTEXT_WINDOW
    breadth_col = "pool_breadth_%s" % POOL_CONTEXT_WINDOW
    median_vol_col = "pool_median_vol_%s" % POOL_CONTEXT_WINDOW
    if ann_col in out.columns:
        out[breadth_col] = float((out[ann_col] > 0).mean())
    if vol_col in out.columns:
        out[median_vol_col] = float(out[vol_col].median())
    return out


def add_rank_features(df):
    out = df.copy()
    for col in RAW_FEATURE_COLS:
        if col in out.columns:
            out["rank_" + col] = out[col].rank(pct=True)
    return out

In [ ]:
# =========================
# 2. Build/load panel and ensure labels
# =========================
def first_existing_path(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def fetch_feature_rows(feature_date, etfs):
    rows = []
    fields = ["open", "high", "low", "close", "volume", "money"]
    for etf_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        try:
            price_df = get_price(etf_chunk, end_date=feature_date, frequency="daily", fields=fields, count=PRICE_HISTORY_COUNT, panel=False, fq="pre", skip_paused=False)
        except Exception as err:
            print("price feature chunk failed", feature_date, err)
            continue
        if price_df is None or price_df.empty:
            continue
        for code, one in price_df.groupby("code"):
            rec = calc_one_etf_features(code, one)
            if rec is not None:
                rows.append(rec)
    if len(rows) == 0:
        return pd.DataFrame()
    return pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)


def fetch_future_returns_multi(feature_date, etfs, horizons):
    horizons = sorted([int(x) for x in horizons])
    if len(etfs) == 0 or len(horizons) == 0:
        return pd.DataFrame()
    max_h = max(horizons)
    try:
        td = pd.to_datetime(get_trade_days(start_date=feature_date, count=max_h + 1))
    except NameError:
        raise RuntimeError("Need JoinQuant API to calculate missing future return labels.")
    except Exception as err:
        print("future trade days failed", feature_date, err)
        return pd.DataFrame()
    if len(td) < max_h + 1:
        return pd.DataFrame()
    rows = []
    for etf_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        try:
            px = get_price(etf_chunk, start_date=feature_date, end_date=pd.Timestamp(td[max_h]).normalize(), frequency="daily", fields=["close"], panel=False, fq="pre", skip_paused=False)
        except Exception as err:
            print("future price chunk failed", feature_date, err)
            continue
        if px is None or px.empty:
            continue
        for code, one in px.groupby("code"):
            one = one.sort_values("time")
            if len(one) < max_h + 1:
                continue
            start_close = float(one.iloc[0]["close"])
            if start_close <= 0:
                continue
            rec = {"code": code}
            for h in horizons:
                end_close = float(one.iloc[h]["close"])
                rec["future_ret_%sd" % h] = end_close / start_close - 1.0
                rec["next_date_%sd" % h] = pd.Timestamp(td[h]).normalize()
            rows.append(rec)
    return pd.DataFrame(rows)


def build_one_feature_date(feature_date):
    etfs = get_etf_universe_on_date(feature_date)
    if len(etfs) == 0:
        return pd.DataFrame()
    feature_df = fetch_feature_rows(feature_date, etfs)
    if feature_df.empty:
        return pd.DataFrame()
    if "money_mean_20" in feature_df.columns:
        feature_df = feature_df[feature_df["money_mean_20"] >= MIN_AVG_MONEY_20].copy()
    if feature_df.empty:
        return pd.DataFrame()
    feature_df = add_pool_context_features(feature_df)
    label_df = fetch_future_returns_multi(feature_date, list(feature_df["code"]), LABEL_HORIZONS)
    if label_df.empty:
        return pd.DataFrame()
    out = feature_df.merge(label_df, on="code", how="inner")
    if out.empty:
        return out
    name_map = ETF_NAME_CACHE.get(str(pd.Timestamp(feature_date).date()), {})
    out["name"] = out["code"].map(name_map).fillna("")
    out["etf_group"] = out.apply(lambda r: classify_etf_group(r["code"], r.get("name", "")), axis=1)
    out = add_rank_features(out)
    out["feature_date"] = pd.Timestamp(feature_date).normalize()
    out["rebalance_date"] = out["feature_date"]
    for h in LABEL_HORIZONS:
        ret_col = "future_ret_%sd" % h
        alpha_col = "target_alpha_%sd" % h
        if ret_col in out.columns:
            out[alpha_col] = out[ret_col] - out[ret_col].median()
    return out


def build_weekly_panel():
    feature_dates = get_weekly_feature_dates(START_DATE, END_DATE)
    print("feature weeks:", len(feature_dates), "from", feature_dates[0] if feature_dates else None, "to", feature_dates[-1] if feature_dates else None)
    parts = []
    for dt in tqdm(feature_dates, desc="build etf weekly panel"):
        one = build_one_feature_date(dt)
        if one is not None and not one.empty:
            parts.append(one)
        if len(parts) > 0 and len(parts) % 20 == 0:
            gc.collect()
    if len(parts) == 0:
        raise RuntimeError("No ETF weekly samples were built. Check JoinQuant data access.")
    panel = pd.concat(parts, ignore_index=True, sort=False)
    return normalize_panel_dates(panel)


def normalize_panel_dates(panel):
    out = drop_duplicate_columns(panel).copy()
    date_cols = ["feature_date", "rebalance_date", "next_date"]
    for h in LABEL_HORIZONS:
        date_cols.append("next_date_%sd" % h)
    for c in date_cols:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c])
    if "next_date" not in out.columns and "next_date_5d" in out.columns:
        out["next_date"] = out["next_date_5d"]
    return out


def ensure_alpha_columns(panel):
    out = panel.copy()
    for h in LABEL_HORIZONS:
        ret_col = "future_ret_%sd" % h
        alpha_col = "target_alpha_%sd" % h
        if ret_col in out.columns and alpha_col not in out.columns:
            out[alpha_col] = out[ret_col] - out.groupby("feature_date")[ret_col].transform("median")
    return out


def ensure_horizon_labels(panel):
    out = normalize_panel_dates(panel)
    missing = []
    for h in LABEL_HORIZONS:
        if "future_ret_%sd" % h not in out.columns:
            missing.append(h)
    if not missing:
        return ensure_alpha_columns(out)
    print("missing horizon labels, calculating via JoinQuant API:", missing)
    additions = []
    base_cols = ["feature_date", "code"]
    for dt, gdf in tqdm(out[base_cols].drop_duplicates().groupby("feature_date"), desc="add future labels"):
        one = fetch_future_returns_multi(dt, sorted(gdf["code"].dropna().unique().tolist()), missing)
        if one is not None and not one.empty:
            one["feature_date"] = pd.Timestamp(dt).normalize()
            additions.append(one)
    if len(additions) == 0:
        print("no labels added; targets needing missing horizons will be skipped")
        return ensure_alpha_columns(out)
    add_df = pd.concat(additions, ignore_index=True, sort=False)
    add_df = drop_duplicate_columns(add_df)
    overlap_cols = [c for c in add_df.columns if c not in ["feature_date", "code"] and c in out.columns]
    if overlap_cols:
        out = out.drop(columns=overlap_cols)
    out = out.merge(add_df, on=["feature_date", "code"], how="left")
    out = drop_duplicate_columns(out)
    return ensure_alpha_columns(normalize_panel_dates(out))


def drop_bad_label_dates(panel):
    out = panel.copy()
    bad_dates = set()
    for h in LABEL_HORIZONS:
        ret_col = "future_ret_%sd" % h
        if ret_col not in out.columns:
            continue
        q = out.groupby("feature_date")[ret_col].agg(["count", lambda s: (s == 0).mean()])
        q.columns = ["sample_count", "zero_return_rate"]
        for dt in q[q["zero_return_rate"] >= 0.98].index:
            bad_dates.add(dt)
    if bad_dates:
        print("drop incomplete label dates:", sorted([str(pd.Timestamp(x).date()) for x in bad_dates]))
        out = out[~out["feature_date"].isin(bad_dates)].copy()
    return out


def load_or_build_panel():
    source = first_existing_path(SOURCE_PANEL_CANDIDATES)
    if REBUILD_DATA is True:
        panel = build_weekly_panel()
    elif REBUILD_DATA is False:
        if source is None:
            raise IOError("No cached panel found. Set REBUILD_DATA=True in JoinQuant environment.")
        print("load cached panel:", source)
        panel = pd.read_csv(source)
    else:
        if source is not None:
            print("load cached panel:", source)
            panel = pd.read_csv(source)
        else:
            panel = build_weekly_panel()
    panel = drop_duplicate_columns(panel)
    panel = ensure_horizon_labels(panel)
    panel = drop_bad_label_dates(panel)
    panel = normalize_panel_dates(panel)
    panel.to_csv(PANEL_CSV, index=False)
    return panel


panel_df = load_or_build_panel()
print("panel:", panel_df.shape)
print(panel_df[["feature_date"]].agg(["min", "max"]))
print("available target cols:", [c for c in panel_df.columns if c.startswith("future_ret_") or c.startswith("target_alpha_")])
display(panel_df.head())

In [ ]:
# =========================
# 3. Train, score, and export model bundles
# =========================
def target_horizon_from_col(target_col):
    m = str(target_col).split("_")[-1]
    if m.endswith("d"):
        try:
            return int(m[:-1])
        except Exception:
            return 5
    return 5


def next_col_for_target(target_col):
    h = target_horizon_from_col(target_col)
    col = "next_date_%sd" % h
    if col in panel_df.columns:
        return col
    return "next_date"


def rebalance_rule_for_target(target_name):
    rule = TARGET_REBALANCE_RULES.get(str(target_name), None)
    if rule is None:
        return "weekly", 1
    mode = str(rule.get("rebalance_mode", "weekly"))
    interval = int(rule.get("rebalance_interval_weeks", 1))
    if interval < 1:
        interval = 1
    return mode, interval


def money_threshold_tag(min_money):
    return "money%dw" % int(round(float(min_money) / 10000.0))


def filter_by_money_threshold(df, min_money):
    if "money_mean_20" not in df.columns:
        raise ValueError("money_mean_20 is required for min_avg_money_20 grid")
    out = df[df["money_mean_20"] >= float(min_money)].copy()
    return out


def clean_train_df(df, target_col):
    df = drop_duplicate_columns(df)
    cols = unique_keep_order(["code", "name", "etf_group", "feature_date", target_col] + FEATURE_COLS)
    cols = [c for c in cols if c in df.columns]
    out = df.loc[:, cols].replace([np.inf, -np.inf], np.nan)
    out = out[~out[target_col].isnull()].copy()
    return out


def train_lgb_or_fallback(train_df, target_col, feature_cols):
    train_df = drop_duplicate_columns(train_df)
    feature_cols = unique_keep_order(feature_cols)
    X_raw = train_df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
    fill_values = X_raw.median().to_dict()
    X = X_raw.fillna(pd.Series(fill_values)).fillna(0).astype(np.float32)
    y = train_df[target_col].astype(np.float32).values
    try:
        import lightgbm as lgb
        dtrain = lgb.Dataset(X[feature_cols], label=y, feature_name=list(feature_cols), free_raw_data=True)
        model = lgb.train(LGB_PARAMS, dtrain, num_boost_round=NUM_BOOST_ROUND)
        backend = "lightgbm"
        del dtrain
    except Exception as err:
        print("LightGBM unavailable or failed, fallback to sklearn RandomForestRegressor:", err)
        from sklearn.ensemble import RandomForestRegressor
        model = RandomForestRegressor(n_estimators=180, max_depth=6, min_samples_leaf=30, random_state=RANDOM_SEED, n_jobs=1)
        model.fit(X[feature_cols], y)
        backend = "sklearn_random_forest"
    del X_raw, X, y
    gc.collect()
    return model, fill_values, backend


def predict_model(model, fill_values, df, feature_cols, chunk_size=12000):
    df = drop_duplicate_columns(df)
    feature_cols = unique_keep_order(feature_cols)
    preds = []
    n = len(df)
    for start in range(0, n, chunk_size):
        part = df.iloc[start:start + chunk_size]
        X_raw = part.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
        X = X_raw.fillna(pd.Series(fill_values)).fillna(0).astype(np.float32)
        preds.append(np.asarray(model.predict(X[feature_cols])).reshape(-1))
        del X_raw, X, part
        gc.collect()
    if len(preds) == 0:
        return np.asarray([])
    out = np.concatenate(preds)
    del preds
    gc.collect()
    return out


def make_bundle(model, fill_values, backend, target_name, target_col, train_tag, train_start, train_end, min_money):
    rebalance_mode, rebalance_interval_weeks = rebalance_rule_for_target(target_name)
    return {
        "objective": "etf_ml_weekly_lgb_v2",
        "research_version": "etf_ml_v11_2021_validation",
        "model": model,
        "backend": backend,
        "target_name": target_name,
        "target_col": target_col,
        "prediction_horizon_days": target_horizon_from_col(target_col),
        "rebalance_mode": rebalance_mode,
        "rebalance_interval_weeks": rebalance_interval_weeks,
        "train_tag": train_tag,
        "train_start": train_start,
        "train_end": train_end,
        "feature_cols": list(FEATURE_COLS),
        "raw_feature_cols": list(RAW_FEATURE_COLS),
        "rank_feature_cols": list(RANK_FEATURE_COLS),
        "context_feature_cols": list(CONTEXT_FEATURE_COLS),
        "fill_values": dict(fill_values),
        "lookback_days": LOOKBACK_DAYS,
        "trend_windows": list(TREND_WINDOWS),
        "trend_weight_end": TREND_WEIGHT_END,
        "price_history_count": PRICE_HISTORY_COUNT,
        "pool_context_window": POOL_CONTEXT_WINDOW,
        "min_listing_days": MIN_LISTING_DAYS,
        "min_avg_money_20": float(min_money),
        "stock_num": 3,
        "max_etf_group_holdings": 1,
        "benchmark": BENCHMARK,
        "exclude_name_keywords": list(EXCLUDE_NAME_KEYWORDS),
    }


def append_csv(df, path):
    if df is None or df.empty:
        return
    write_header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=write_header, index=False)


def build_score_export_cols():
    target_cols = []
    for target_name, target_col, eval_ret_col in TARGET_SPECS:
        target_cols.append(target_col)
        target_cols.append(eval_ret_col)
    next_cols = ["next_date"]
    for h in LABEL_HORIZONS:
        next_cols.append("next_date_%sd" % h)
    meta_cols = [
        "model_name", "model_file", "target_name", "target_col", "prediction_horizon_days",
        "rebalance_mode", "rebalance_interval_weeks", "min_avg_money_20", "money_threshold_tag",
        "eval_ret_col", "next_col", "train_tag", "train_start", "train_end", "oos_start_date",
        "code", "name", "etf_group", "feature_date", "rebalance_date",
        "score", "eval_ret",
    ]
    # Score panel is for proxy evaluation only; do not write the 96 feature columns.
    return unique_keep_order(meta_cols + next_cols + target_cols)


SCORE_EXPORT_COLS = build_score_export_cols()


def append_score_csv(df, path):
    if df is None or df.empty:
        return
    out = drop_duplicate_columns(df).reindex(columns=SCORE_EXPORT_COLS)
    write_header = not os.path.exists(path)
    out.to_csv(path, mode="a", header=write_header, index=False)


for _p in [SCORE_CSV, MODEL_MANIFEST_CSV]:
    if os.path.exists(_p):
        os.remove(_p)

manifest_rows = []
score_export_cols = None
available_specs = []
for spec in TARGET_SPECS:
    target_name, target_col, eval_ret_col = spec
    if target_col in panel_df.columns and eval_ret_col in panel_df.columns:
        available_specs.append(spec)
    else:
        print("skip missing target spec:", spec)

for min_money in tqdm(MIN_AVG_MONEY_20_GRID, desc="money thresholds"):
    money_tag = money_threshold_tag(min_money)
    panel_money_df = filter_by_money_threshold(panel_df, min_money)
    if panel_money_df.empty:
        print("skip empty money threshold", min_money)
        continue
    print("money threshold", int(min_money), "rows", len(panel_money_df), "weeks", panel_money_df["feature_date"].nunique())
    for train_tag, train_start, train_end in tqdm(TRAIN_WINDOWS, desc=money_tag, leave=False):
        train_start_ts = pd.Timestamp(train_start)
        train_end_ts = pd.Timestamp(train_end)
        for target_name, target_col, eval_ret_col in tqdm(available_specs, desc=train_tag, leave=False):
            next_col = next_col_for_target(target_col)
            if next_col not in panel_money_df.columns:
                print("skip target next_col missing", target_col, next_col)
                continue
            train_mask = (panel_money_df["feature_date"] >= train_start_ts) & (panel_money_df[next_col] <= train_end_ts)
            # Deployment-parity OOS: the model is only available after train_end.
            # Do not score feature dates inside the training window just because their label exits after train_end.
            score_mask = panel_money_df["feature_date"] > train_end_ts
            train_df = clean_train_df(panel_money_df.loc[train_mask], target_col)
            if train_df.empty:
                print("skip empty train", money_tag, train_tag, target_col)
                continue
            score_base_cols = unique_keep_order(["code", "name", "etf_group", "feature_date", next_col, target_col, eval_ret_col] + FEATURE_COLS)
            score_base_cols = [c for c in score_base_cols if c in panel_money_df.columns]
            score_base = drop_duplicate_columns(panel_money_df.loc[score_mask, score_base_cols].copy())
            if score_base.empty:
                print("skip empty score", money_tag, train_tag, target_col)
                continue
            print("training", money_tag, train_tag, target_name, "samples", len(train_df), "weeks", train_df["feature_date"].nunique(), "score", len(score_base))
            model, fill_values, backend = train_lgb_or_fallback(train_df, target_col, FEATURE_COLS)
            bundle = make_bundle(model, fill_values, backend, target_name, target_col, train_tag, train_start, train_end, min_money)
            model_file = "model_etf_ml_v11_lgb_%s_%s_%s.pkl" % (target_name, train_tag, money_tag)
            model_path = os.path.join(MODEL_DIR, model_file)
            with open(model_path, "wb") as f:
                pickle.dump(bundle, f, protocol=2)
            model_name = "%s_%s_%s" % (target_name, train_tag, money_tag)
            score_pred = predict_model(model, fill_values, score_base, FEATURE_COLS)
            export_cols = unique_keep_order(["code", "name", "etf_group", "feature_date", "rebalance_date", next_col, target_col, eval_ret_col])
            export_cols = [c for c in export_cols if c in score_base.columns]
            scored = score_base.loc[:, export_cols].copy()
            scored["score"] = score_pred
            scored["model_name"] = model_name
            scored["model_file"] = os.path.join("models", model_file)
            scored["target_name"] = target_name
            scored["target_col"] = target_col
            scored["prediction_horizon_days"] = target_horizon_from_col(target_col)
            scored["rebalance_mode"], scored["rebalance_interval_weeks"] = rebalance_rule_for_target(target_name)
            scored["min_avg_money_20"] = float(min_money)
            scored["money_threshold_tag"] = money_tag
            scored["eval_ret_col"] = eval_ret_col
            scored["next_col"] = next_col
            scored["train_tag"] = train_tag
            scored["train_start"] = train_start
            scored["train_end"] = train_end
            scored["oos_start_date"] = pd.Timestamp(train_end)
            scored["eval_ret"] = scored[eval_ret_col]
            append_score_csv(scored, SCORE_CSV)
            del score_pred
            manifest_rows.append({
                "model_name": model_name,
                "model_file": os.path.join("models", model_file),
                "backend": backend,
                "target_name": target_name,
                "target_col": target_col,
                "prediction_horizon_days": target_horizon_from_col(target_col),
                "rebalance_mode": rebalance_rule_for_target(target_name)[0],
                "rebalance_interval_weeks": rebalance_rule_for_target(target_name)[1],
                "min_avg_money_20": float(min_money),
                "money_threshold_tag": money_tag,
                "eval_ret_col": eval_ret_col,
                "next_col": next_col,
                "train_tag": train_tag,
                "train_start": train_start,
                "train_end": train_end,
                "oos_start_date": train_end,
                "train_samples": int(len(train_df)),
                "train_weeks": int(train_df["feature_date"].nunique()),
                "score_samples": int(len(scored)),
                "score_weeks": int(scored["feature_date"].nunique()),
                "feature_count": int(len(FEATURE_COLS)),
            })
            del scored, model, bundle, train_df, score_base, fill_values
            gc.collect()
    del panel_money_df
    gc.collect()

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MODEL_MANIFEST_CSV, index=False)
print("manifest:", manifest_df.shape)
display(manifest_df)

score_df = pd.read_csv(SCORE_CSV) if os.path.exists(SCORE_CSV) else pd.DataFrame()
for _c in ["feature_date", "train_start", "train_end", "oos_start_date"]:
    if _c in score_df.columns:
        score_df[_c] = pd.to_datetime(score_df[_c])
print("score:", score_df.shape)
display(score_df.head())

In [ ]:
# =========================
# 4. Close-label proxy: topN and theme cap comparison
# =========================
def summarize_returns(ret_series):
    r = pd.Series(ret_series).dropna().astype(float)
    if len(r) == 0:
        return {"weeks": 0}
    nav = (1.0 + r).cumprod()
    dd = nav / nav.cummax() - 1.0
    std = float(r.std())
    return {
        "periods": int(len(r)),
        "weeks": int(len(r)),  # backward-compatible name; mixed cadence experiments should read periods.
        "cum_ret": float(nav.iloc[-1] - 1.0),
        "mean_period_ret": float(r.mean()),
        "mean_weekly_ret": float(r.mean()),
        "win_rate": float((r > 0).mean()),
        "weekly_sharpe": float(r.mean() / std) if std > 0 else np.nan,
        "max_drawdown": float(dd.min()),
    }


def select_targets(gdf, top_n, group_cap):
    x = gdf.sort_values("score", ascending=False).copy()
    if group_cap <= 0:
        return x.head(top_n)
    selected_idx = []
    group_count = {}
    for idx, row in x.iterrows():
        group_name = row.get("etf_group", "")
        if pd.isnull(group_name) or str(group_name) == "":
            group_name = "single_" + str(row.get("code", idx))
        cnt = group_count.get(group_name, 0)
        if cnt >= group_cap:
            continue
        selected_idx.append(idx)
        group_count[group_name] = cnt + 1
        if len(selected_idx) >= top_n:
            break
    if len(selected_idx) < top_n:
        for idx, _ in x.iterrows():
            if idx not in selected_idx:
                selected_idx.append(idx)
            if len(selected_idx) >= top_n:
                break
    return x.loc[selected_idx]


def scheduled_score_data(score_data):
    if score_data.empty:
        return score_data
    parts = []
    for model_name, gdf0 in score_data.groupby("model_name"):
        gdf = gdf0.copy()
        gdf["feature_date"] = pd.to_datetime(gdf["feature_date"])
        mode = str(gdf["rebalance_mode"].iloc[0]) if "rebalance_mode" in gdf.columns else "weekly"
        interval = int(gdf["rebalance_interval_weeks"].iloc[0]) if "rebalance_interval_weeks" in gdf.columns else 1
        if interval < 1:
            interval = 1
        dates = sorted(gdf["feature_date"].dropna().unique())
        if mode == "monthly":
            keep_dates = []
            seen_months = set()
            for dt in dates:
                month_key = pd.Timestamp(dt).strftime("%Y-%m")
                if month_key in seen_months:
                    continue
                seen_months.add(month_key)
                keep_dates.append(pd.Timestamp(dt))
        else:
            keep_dates = [pd.Timestamp(dt) for i, dt in enumerate(dates) if i % interval == 0]
        if len(keep_dates) == 0:
            continue
        keep_set = set(keep_dates)
        parts.append(gdf[gdf["feature_date"].isin(keep_set)].copy())
    if len(parts) == 0:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True, sort=False)


def build_weekly_proxy(score_data):
    rows = []
    rng = np.random.RandomState(RANDOM_SEED)
    if score_data.empty:
        return pd.DataFrame()
    scheduled = scheduled_score_data(score_data)
    print("scheduled score rows:", scheduled.shape, "raw:", score_data.shape)
    for (model_name, dt), gdf in tqdm(scheduled.groupby(["model_name", "feature_date"]), desc="scheduled proxy"):
        gdf = gdf.dropna(subset=["eval_ret"]).copy()
        if gdf.empty:
            continue
        for top_n in TOP_N_LIST:
            for group_cap in GROUP_CAP_LIST:
                one = select_targets(gdf, min(top_n, len(gdf)), group_cap)
                if one.empty:
                    continue
                rand_n = min(top_n, len(gdf))
                random_ret = float(gdf.sample(rand_n, random_state=int(rng.randint(0, 1000000)))["eval_ret"].mean())
                rows.append({
                    "model_name": model_name,
                    "model_file": one["model_file"].iloc[0],
                    "target_name": one["target_name"].iloc[0],
                    "target_col": one["target_col"].iloc[0],
                    "prediction_horizon_days": int(one.get("prediction_horizon_days", pd.Series([np.nan])).iloc[0]) if not pd.isnull(one.get("prediction_horizon_days", pd.Series([np.nan])).iloc[0]) else np.nan,
                    "rebalance_mode": one.get("rebalance_mode", pd.Series(["weekly"])).iloc[0],
                    "rebalance_interval_weeks": int(one.get("rebalance_interval_weeks", pd.Series([1])).iloc[0]),
                    "min_avg_money_20": float(one.get("min_avg_money_20", pd.Series([np.nan])).iloc[0]) if not pd.isnull(one.get("min_avg_money_20", pd.Series([np.nan])).iloc[0]) else np.nan,
                    "money_threshold_tag": one.get("money_threshold_tag", pd.Series([""])).iloc[0],
                    "eval_ret_col": one["eval_ret_col"].iloc[0],
                    "train_tag": one["train_tag"].iloc[0],
                    "train_start": one["train_start"].iloc[0],
                    "train_end": one["train_end"].iloc[0],
                    "oos_start_date": one.get("oos_start_date", pd.Series([one["train_end"].iloc[0]])).iloc[0],
                    "feature_date": dt,
                    "year": pd.Timestamp(dt).year,
                    "top_n": int(top_n),
                    "group_cap": int(group_cap),
                    "portfolio_ret": float(one["eval_ret"].mean()),
                    "median_ret": float(gdf["eval_ret"].median()),
                    "random_ret": random_ret,
                    "target_count": int(len(one)),
                    "targets": "|".join(one["code"].astype(str).tolist()),
                    "target_names": "|".join(one.get("name", pd.Series([""] * len(one))).astype(str).tolist()),
                    "target_groups": "|".join(one.get("etf_group", pd.Series([""] * len(one))).astype(str).tolist()),
                })
    return pd.DataFrame(rows)


def build_summary(weekly_df):
    rows = []
    if weekly_df.empty:
        return pd.DataFrame()
    group_cols = ["model_name", "model_file", "target_name", "target_col", "eval_ret_col", "train_tag", "oos_start_date", "rebalance_mode", "rebalance_interval_weeks", "min_avg_money_20", "money_threshold_tag", "top_n", "group_cap"]
    for keys, gdf in weekly_df.groupby(group_cols):
        rec = dict(zip(group_cols, keys))
        rec.update(summarize_returns(gdf["portfolio_ret"]))
        rec["cum_excess_median"] = float((1.0 + (gdf["portfolio_ret"] - gdf["median_ret"])).prod() - 1.0)
        rec["cum_excess_random"] = float((1.0 + (gdf["portfolio_ret"] - gdf["random_ret"])).prod() - 1.0)
        rows.append(rec)
    return pd.DataFrame(rows).sort_values(["top_n", "group_cap", "cum_ret"], ascending=[True, True, False])


def build_yearly(weekly_df):
    rows = []
    if weekly_df.empty:
        return pd.DataFrame()
    group_cols = ["model_name", "model_file", "target_name", "train_tag", "oos_start_date", "rebalance_mode", "rebalance_interval_weeks", "min_avg_money_20", "money_threshold_tag", "top_n", "group_cap", "year"]
    for keys, gdf in weekly_df.groupby(group_cols):
        rec = dict(zip(group_cols, keys))
        rec.update(summarize_returns(gdf["portfolio_ret"]))
        rec["cum_excess_median"] = float((1.0 + (gdf["portfolio_ret"] - gdf["median_ret"])).prod() - 1.0)
        rec["cum_excess_random"] = float((1.0 + (gdf["portfolio_ret"] - gdf["random_ret"])).prod() - 1.0)
        rows.append(rec)
    return pd.DataFrame(rows).sort_values(["top_n", "group_cap", "year", "cum_ret"], ascending=[True, True, True, False])


def build_money_threshold_summary(summary_data):
    if summary_data.empty or "min_avg_money_20" not in summary_data.columns:
        return pd.DataFrame()
    rows = []
    for min_money, gdf in summary_data.groupby("min_avg_money_20"):
        g = gdf.sort_values("cum_ret", ascending=False).copy()
        best = g.iloc[0]
        row = {
            "min_avg_money_20": float(min_money),
            "best_model_name": best.get("model_name", ""),
            "best_target_name": best.get("target_name", ""),
            "best_train_tag": best.get("train_tag", ""),
            "best_top_n": int(best.get("top_n", 0)),
            "best_group_cap": int(best.get("group_cap", 0)),
            "best_cum_ret": float(best.get("cum_ret", np.nan)),
            "best_max_drawdown": float(best.get("max_drawdown", np.nan)),
            "best_cum_excess_random": float(best.get("cum_excess_random", np.nan)),
            "model_count": int(len(gdf)),
        }
        focus = gdf[(gdf["top_n"] == 1) & (gdf["group_cap"] == 0)].copy()
        if not focus.empty:
            focus_best = focus.sort_values("cum_ret", ascending=False).iloc[0]
            row["top1_nocap_best_model"] = focus_best.get("model_name", "")
            row["top1_nocap_best_cum_ret"] = float(focus_best.get("cum_ret", np.nan))
            row["top1_nocap_best_mdd"] = float(focus_best.get("max_drawdown", np.nan))
        rows.append(row)
    return pd.DataFrame(rows).sort_values("min_avg_money_20")


def build_candidate_models(summary_data, yearly_data):
    if summary_data.empty:
        return pd.DataFrame()
    s = summary_data.copy()
    for col in ["cum_ret", "cum_excess_random", "max_drawdown", "win_rate"]:
        if col in s.columns:
            s[col] = pd.to_numeric(s[col], errors="coerce")
    # Prefer the actual current use case: no theme cap. Keep top1/top3 rows as deployment configs.
    if "group_cap" in s.columns:
        preferred = s[s["group_cap"] == 0].copy()
        if preferred.empty:
            preferred = s.copy()
    else:
        preferred = s.copy()
    preferred["candidate_score"] = (
        preferred["cum_ret"].fillna(-999.0)
        + 0.5 * preferred.get("cum_excess_random", pd.Series(0.0, index=preferred.index)).fillna(0.0)
        + preferred.get("max_drawdown", pd.Series(0.0, index=preferred.index)).fillna(0.0)
    )

    rows = []
    for min_money, gdf in preferred.groupby("min_avg_money_20"):
        g = gdf.sort_values(["candidate_score", "cum_ret"], ascending=[False, False]).head(CANDIDATE_MAX_PER_MONEY)
        rows.append(g)
    if len(rows) == 0:
        return pd.DataFrame()
    cand = pd.concat(rows, ignore_index=True, sort=False)
    cand = cand.sort_values(["candidate_score", "cum_ret"], ascending=[False, False]).head(CANDIDATE_MAX_OVERALL).copy()

    if yearly_data is not None and not yearly_data.empty:
        y = yearly_data.copy()
        key_cols = ["model_name", "top_n", "group_cap"]
        y_stats = []
        for keys, gdf in y.groupby(key_cols):
            rec = dict(zip(key_cols, keys))
            ret = pd.to_numeric(gdf["cum_ret"], errors="coerce").dropna()
            rec["year_count"] = int(len(ret))
            rec["positive_year_count"] = int((ret > 0).sum())
            rec["min_year_cum_ret"] = float(ret.min()) if len(ret) else np.nan
            rec["mean_year_cum_ret"] = float(ret.mean()) if len(ret) else np.nan
            y_stats.append(rec)
        y_stats_df = pd.DataFrame(y_stats)
        if not y_stats_df.empty:
            cand = cand.merge(y_stats_df, on=key_cols, how="left")

    cand["suggested_MODEL_FILE"] = cand["model_file"].astype(str)
    cand["suggested_STOCK_NUM_OVERRIDE"] = cand["top_n"].astype(int)
    cand["suggested_MAX_ETF_GROUP_HOLDINGS_OVERRIDE"] = cand["group_cap"].astype(int)
    cand.loc[cand["suggested_MAX_ETF_GROUP_HOLDINGS_OVERRIDE"] <= 0, "suggested_MAX_ETF_GROUP_HOLDINGS_OVERRIDE"] = 0
    out_cols = [
        "model_name", "model_file", "target_name", "train_tag", "min_avg_money_20", "money_threshold_tag",
        "top_n", "group_cap", "rebalance_mode", "rebalance_interval_weeks",
        "cum_ret", "cum_excess_random", "max_drawdown", "win_rate", "candidate_score",
        "year_count", "positive_year_count", "min_year_cum_ret", "mean_year_cum_ret",
        "suggested_MODEL_FILE", "suggested_STOCK_NUM_OVERRIDE", "suggested_MAX_ETF_GROUP_HOLDINGS_OVERRIDE",
    ]
    out_cols = [c for c in out_cols if c in cand.columns]
    return cand.loc[:, out_cols].reset_index(drop=True)


def export_candidate_model_files(candidate_df):
    if candidate_df is None or candidate_df.empty:
        return []
    copied = []
    if not os.path.exists(CANDIDATE_MODEL_DIR):
        os.makedirs(CANDIDATE_MODEL_DIR)
    for _, row in candidate_df.iterrows():
        rel_path = str(row.get("model_file", ""))
        if rel_path == "":
            continue
        src_path = os.path.join(OUT_DIR, rel_path)
        if not os.path.exists(src_path):
            print("candidate model file missing:", src_path)
            continue
        dst_path = os.path.join(CANDIDATE_MODEL_DIR, os.path.basename(src_path))
        shutil.copy2(src_path, dst_path)
        copied.append(dst_path)
    print("copied candidate model files:", len(copied), "->", CANDIDATE_MODEL_DIR)
    return copied


weekly_proxy_df = build_weekly_proxy(score_df)
summary_df = build_summary(weekly_proxy_df)
yearly_df = build_yearly(weekly_proxy_df)
money_threshold_summary_df = build_money_threshold_summary(summary_df)
candidate_models_df = build_candidate_models(summary_df, yearly_df)
copied_candidate_model_files = export_candidate_model_files(candidate_models_df)
weekly_proxy_df.to_csv(WEEKLY_PROXY_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)
yearly_df.to_csv(YEARLY_CSV, index=False)
money_threshold_summary_df.to_csv(MONEY_THRESHOLD_SUMMARY_CSV, index=False)
candidate_models_df.to_csv(CANDIDATE_MODELS_CSV, index=False)

print("weekly proxy:", weekly_proxy_df.shape)
print("summary:", summary_df.shape)
print("yearly:", yearly_df.shape)
print("money threshold summary:", money_threshold_summary_df.shape)
print("candidate models:", candidate_models_df.shape)
print("candidate model dir:", CANDIDATE_MODEL_DIR)
display(money_threshold_summary_df)
display(candidate_models_df)
display(summary_df.head(40))
display(yearly_df.head(80))

In [ ]:
# =========================
# 5. Stability diagnostics: overlap, score gap, and drawdown events
# =========================
def build_target_overlap(weekly_df):
    rows = []
    if weekly_df is None or weekly_df.empty:
        return pd.DataFrame()
    base = weekly_df[(weekly_df["top_n"] == 1) & (weekly_df["group_cap"] == 0)].copy()
    if base.empty:
        return pd.DataFrame()
    keys = sorted(base["model_name"].dropna().unique())
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            a_name = keys[i]
            b_name = keys[j]
            a = base[base["model_name"] == a_name][["feature_date", "targets", "portfolio_ret"]].copy()
            b = base[base["model_name"] == b_name][["feature_date", "targets", "portfolio_ret"]].copy()
            a = a.rename(columns={"targets": "target_a", "portfolio_ret": "ret_a"})
            b = b.rename(columns={"targets": "target_b", "portfolio_ret": "ret_b"})
            m = a.merge(b, on="feature_date", how="inner")
            if m.empty:
                continue
            same = (m["target_a"].astype(str) == m["target_b"].astype(str))
            rows.append({
                "model_a": a_name,
                "model_b": b_name,
                "common_periods": int(len(m)),
                "target_overlap_rate": float(same.mean()),
                "same_target_count": int(same.sum()),
                "ret_corr": float(m[["ret_a", "ret_b"]].corr().iloc[0, 1]) if len(m) >= 3 else np.nan,
                "cum_ret_a_common": float((1.0 + m["ret_a"].astype(float)).prod() - 1.0),
                "cum_ret_b_common": float((1.0 + m["ret_b"].astype(float)).prod() - 1.0),
            })
    return pd.DataFrame(rows).sort_values(["common_periods", "target_overlap_rate"], ascending=[False, False]) if rows else pd.DataFrame()


def build_score_gap_diagnostics(score_data):
    rows = []
    if score_data is None or score_data.empty:
        return pd.DataFrame()
    data = scheduled_score_data(score_data)
    for (model_name, feature_date), gdf0 in data.groupby(["model_name", "feature_date"]):
        gdf = gdf0.sort_values("score", ascending=False).copy()
        if len(gdf) == 0:
            continue
        top1 = gdf.iloc[0]
        top2_score = float(gdf.iloc[1]["score"]) if len(gdf) >= 2 else np.nan
        top3_score = float(gdf.iloc[2]["score"]) if len(gdf) >= 3 else np.nan
        top1_score = float(top1["score"])
        rows.append({
            "model_name": model_name,
            "feature_date": feature_date,
            "target_name": top1.get("target_name", ""),
            "train_tag": top1.get("train_tag", ""),
            "min_avg_money_20": top1.get("min_avg_money_20", np.nan),
            "top1_code": top1.get("code", ""),
            "top1_name": top1.get("name", ""),
            "top1_group": top1.get("etf_group", ""),
            "top1_score": top1_score,
            "top2_score": top2_score,
            "top3_score": top3_score,
            "gap_1_2": top1_score - top2_score if not pd.isnull(top2_score) else np.nan,
            "gap_1_3": top1_score - top3_score if not pd.isnull(top3_score) else np.nan,
            "score_std_top10": float(gdf.head(10)["score"].std()) if len(gdf) >= 3 else np.nan,
            "top1_eval_ret": top1.get("eval_ret", np.nan),
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["model_name", "feature_date"])


def build_drawdown_events(weekly_df, top_n=1, group_cap=0, worst_n=8):
    rows = []
    if weekly_df is None or weekly_df.empty:
        return pd.DataFrame()
    data = weekly_df[(weekly_df["top_n"] == top_n) & (weekly_df["group_cap"] == group_cap)].copy()
    for model_name, gdf0 in data.groupby("model_name"):
        gdf = gdf0.sort_values("feature_date").copy()
        if gdf.empty:
            continue
        gdf["nav"] = (1.0 + gdf["portfolio_ret"].astype(float)).cumprod()
        gdf["peak"] = gdf["nav"].cummax()
        gdf["drawdown"] = gdf["nav"] / gdf["peak"] - 1.0
        trough_idx = gdf["drawdown"].idxmin()
        peak_idx = gdf.loc[:trough_idx, "nav"].idxmax()
        peak_date = gdf.loc[peak_idx, "feature_date"]
        trough_date = gdf.loc[trough_idx, "feature_date"]
        rows.append({
            "event_type": "max_drawdown",
            "model_name": model_name,
            "top_n": top_n,
            "group_cap": group_cap,
            "feature_date": trough_date,
            "peak_date": peak_date,
            "trough_date": trough_date,
            "portfolio_ret": gdf.loc[trough_idx, "portfolio_ret"],
            "median_ret": gdf.loc[trough_idx, "median_ret"],
            "random_ret": gdf.loc[trough_idx, "random_ret"],
            "drawdown": gdf.loc[trough_idx, "drawdown"],
            "targets": gdf.loc[trough_idx, "targets"],
            "target_names": gdf.loc[trough_idx, "target_names"],
            "target_groups": gdf.loc[trough_idx, "target_groups"],
        })
        worst = gdf.sort_values("portfolio_ret").head(worst_n)
        for _, row in worst.iterrows():
            rows.append({
                "event_type": "worst_period",
                "model_name": model_name,
                "top_n": top_n,
                "group_cap": group_cap,
                "feature_date": row.get("feature_date", ""),
                "peak_date": "",
                "trough_date": "",
                "portfolio_ret": row.get("portfolio_ret", np.nan),
                "median_ret": row.get("median_ret", np.nan),
                "random_ret": row.get("random_ret", np.nan),
                "drawdown": row.get("drawdown", np.nan),
                "targets": row.get("targets", ""),
                "target_names": row.get("target_names", ""),
                "target_groups": row.get("target_groups", ""),
            })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["model_name", "event_type", "portfolio_ret"])


target_overlap_df = build_target_overlap(weekly_proxy_df)
score_gap_diagnostics_df = build_score_gap_diagnostics(score_df)
drawdown_events_df = build_drawdown_events(weekly_proxy_df, top_n=1, group_cap=0, worst_n=8)

target_overlap_df.to_csv(TARGET_OVERLAP_CSV, index=False)
score_gap_diagnostics_df.to_csv(SCORE_GAP_DIAGNOSTICS_CSV, index=False)
drawdown_events_df.to_csv(DRAWDOWN_EVENTS_CSV, index=False)

print("target overlap:", target_overlap_df.shape)
display(target_overlap_df.head(40))
print("score gap diagnostics:", score_gap_diagnostics_df.shape)
display(score_gap_diagnostics_df.head(40))
print("drawdown events:", drawdown_events_df.shape)
display(drawdown_events_df.head(80))

In [ ]:
# =========================
# 7. V11 cost, turnover, common-OOS, and baseline diagnostics
# =========================
V11_COST_WEEKLY_CSV = os.path.join(OUT_DIR, "etf_ml_v11_weekly_cost_proxy.csv")
V11_COST_SUMMARY_CSV = os.path.join(OUT_DIR, "etf_ml_v11_cost_summary.csv")
V11_COMMON_OOS_CSV = os.path.join(OUT_DIR, "etf_ml_v11_common_oos.csv")
V11_STABILITY_CSV = os.path.join(OUT_DIR, "etf_ml_v11_cost_stability.csv")
V11_REPORT_MD = os.path.join(OUT_DIR, "etf_ml_v11_report.md")
RANDOM_BASELINE_SEEDS = list(range(30))
COMMON_OOS_STARTS = ["2025-01-01", "2026-01-01"]


def _parse_targets(x):
    if pd.isnull(x):
        return []
    return [z for z in str(x).split("|") if z]


def _weights_from_targets(targets):
    codes = _parse_targets(targets)
    if not codes:
        return {}
    w = 1.0 / len(codes)
    return {c: w for c in codes}


def _turnover(prev_w, new_w):
    keys = set(prev_w.keys()) | set(new_w.keys())
    traded = float(sum(abs(new_w.get(k, 0.0) - prev_w.get(k, 0.0)) for k in keys))
    return traded, traded / 2.0


def _baseline_top_ret(gdf, factor_col, top_n):
    if factor_col not in gdf.columns:
        return np.nan
    x = gdf.dropna(subset=[factor_col, "eval_ret"]).copy()
    if x.empty:
        return np.nan
    return float(x.sort_values(factor_col, ascending=False).head(min(top_n, len(x)))["eval_ret"].mean())


def _random_baseline_stats(gdf, top_n):
    x = gdf.dropna(subset=["eval_ret"]).copy()
    if x.empty:
        return {"random_mean_ret": np.nan, "random_median_ret": np.nan, "random_min_ret": np.nan, "random_max_ret": np.nan}
    n = min(top_n, len(x))
    vals = [float(x.sample(n, random_state=seed)["eval_ret"].mean()) for seed in RANDOM_BASELINE_SEEDS]
    s = pd.Series(vals)
    return {"random_mean_ret": float(s.mean()), "random_median_ret": float(s.median()), "random_min_ret": float(s.min()), "random_max_ret": float(s.max())}


def build_v11_cost_weekly(weekly_proxy, score_data):
    if weekly_proxy is None or weekly_proxy.empty:
        return pd.DataFrame()
    out_parts = []
    score_scheduled = scheduled_score_data(score_data) if score_data is not None and not score_data.empty else pd.DataFrame()
    score_scheduled["feature_date"] = pd.to_datetime(score_scheduled["feature_date"]) if not score_scheduled.empty else pd.Series(dtype="datetime64[ns]")
    for (model_name, top_n, group_cap), gdf0 in weekly_proxy.groupby(["model_name", "top_n", "group_cap"]):
        gdf = gdf0.sort_values("feature_date").copy()
        prev_w = {}
        rows = []
        for _, row in gdf.iterrows():
            new_w = _weights_from_targets(row.get("targets", ""))
            traded, one_way = _turnover(prev_w, new_w)
            prev_w = new_w
            gross_ret = float(row.get("portfolio_ret", np.nan))
            cost_drag = traded * COST_RATE
            rec = row.to_dict()
            rec["gross_ret"] = gross_ret
            rec["turnover_traded_notional"] = traded
            rec["one_way_turnover"] = one_way
            rec["cost_rate"] = COST_RATE
            rec["cost_drag"] = cost_drag
            rec["net_ret"] = gross_ret - cost_drag if pd.notnull(gross_ret) else np.nan
            score_week = score_scheduled[(score_scheduled["model_name"] == model_name) & (score_scheduled["feature_date"] == pd.Timestamp(row["feature_date"]))]
            if score_week.empty:
                rec.update({"random_mean_ret": np.nan, "random_median_ret": np.nan, "random_min_ret": np.nan, "random_max_ret": np.nan})
                rec["ret20_topn_ret"] = np.nan
                rec["trend25_topn_ret"] = np.nan
            else:
                rec.update(_random_baseline_stats(score_week, int(top_n)))
                rec["ret20_topn_ret"] = _baseline_top_ret(score_week, "ret_20", int(top_n))
                rec["trend25_topn_ret"] = _baseline_top_ret(score_week, "trend_score_25", int(top_n))
            rows.append(rec)
        out_parts.append(pd.DataFrame(rows))
    out = pd.concat(out_parts, ignore_index=True, sort=False) if out_parts else pd.DataFrame()
    out.to_csv(V11_COST_WEEKLY_CSV, index=False)
    return out


def _summary_with_prefix(ret_series, prefix):
    s = summarize_returns(ret_series)
    return {prefix + "_" + k: v for k, v in s.items()}


def build_v11_cost_summary(cost_weekly):
    rows = []
    if cost_weekly is None or cost_weekly.empty:
        return pd.DataFrame()
    group_cols = ["model_name", "target_name", "train_tag", "top_n", "group_cap", "min_avg_money_20", "money_threshold_tag"]
    for keys, gdf in cost_weekly.groupby(group_cols, dropna=False):
        rec = dict(zip(group_cols, keys))
        rec.update(_summary_with_prefix(gdf["net_ret"], "net"))
        rec.update(_summary_with_prefix(gdf["gross_ret"], "gross"))
        rec.update(_summary_with_prefix(gdf["median_ret"], "median"))
        rec.update(_summary_with_prefix(gdf["random_mean_ret"], "random"))
        rec.update(_summary_with_prefix(gdf["ret20_topn_ret"], "ret20"))
        rec.update(_summary_with_prefix(gdf["trend25_topn_ret"], "trend25"))
        rec["avg_turnover_traded_notional"] = float(gdf["turnover_traded_notional"].mean())
        rec["avg_one_way_turnover"] = float(gdf["one_way_turnover"].mean())
        rec["total_cost_drag"] = float(gdf["cost_drag"].sum())
        rec["net_excess_vs_median"] = rec.get("net_cum_ret", np.nan) - rec.get("median_cum_ret", np.nan)
        rec["net_excess_vs_random"] = rec.get("net_cum_ret", np.nan) - rec.get("random_cum_ret", np.nan)
        rec["net_excess_vs_ret20"] = rec.get("net_cum_ret", np.nan) - rec.get("ret20_cum_ret", np.nan)
        rec["net_excess_vs_trend25"] = rec.get("net_cum_ret", np.nan) - rec.get("trend25_cum_ret", np.nan)
        rows.append(rec)
    out = pd.DataFrame(rows).sort_values(["top_n", "net_cum_ret"], ascending=[True, False])
    out.to_csv(V11_COST_SUMMARY_CSV, index=False)
    return out


def build_v11_common_oos(cost_weekly):
    rows = []
    if cost_weekly is None or cost_weekly.empty:
        return pd.DataFrame()
    data = cost_weekly.copy()
    data["feature_date"] = pd.to_datetime(data["feature_date"])
    for start in COMMON_OOS_STARTS:
        sub = data[data["feature_date"] >= pd.Timestamp(start)].copy()
        if sub.empty:
            continue
        for (top_n, target_name), top_df in sub.groupby(["top_n", "target_name"], dropna=False):
            date_sets = [set(pd.to_datetime(g["feature_date"]).dt.normalize()) for _, g in top_df.groupby("model_name")]
            if not date_sets:
                continue
            common_dates = set.intersection(*date_sets) if len(date_sets) > 1 else date_sets[0]
            if not common_dates:
                continue
            common = top_df[top_df["feature_date"].isin(common_dates)].copy()
            for model_name, gdf in common.groupby("model_name"):
                rec = {"common_oos_start": start, "model_name": model_name, "top_n": int(top_n), "target_name": target_name, "common_weeks": len(common_dates)}
                rec.update(_summary_with_prefix(gdf["net_ret"], "net"))
                rec.update(_summary_with_prefix(gdf["gross_ret"], "gross"))
                rec.update(_summary_with_prefix(gdf["median_ret"], "median"))
                rec.update(_summary_with_prefix(gdf["random_mean_ret"], "random"))
                rows.append(rec)
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["common_oos_start", "top_n", "net_cum_ret"], ascending=[True, True, False])
    out.to_csv(V11_COMMON_OOS_CSV, index=False)
    return out


def build_v11_cost_stability(cost_weekly):
    rows = []
    if cost_weekly is None or cost_weekly.empty:
        return pd.DataFrame()
    for top_n, top_df in cost_weekly.groupby("top_n"):
        models = sorted(top_df["model_name"].dropna().unique())
        for i in range(len(models)):
            for j in range(i + 1, len(models)):
                a = top_df[top_df["model_name"] == models[i]][["feature_date", "targets", "net_ret"]].copy()
                b = top_df[top_df["model_name"] == models[j]][["feature_date", "targets", "net_ret"]].copy()
                m = a.merge(b, on="feature_date", suffixes=("_a", "_b"))
                if m.empty:
                    continue
                overlaps = []
                for _, r in m.iterrows():
                    sa = set(_parse_targets(r["targets_a"]))
                    sb = set(_parse_targets(r["targets_b"]))
                    denom = max(1, min(len(sa), len(sb)))
                    overlaps.append(len(sa & sb) / denom)
                rows.append({
                    "top_n": int(top_n),
                    "model_a": models[i],
                    "model_b": models[j],
                    "common_periods": int(len(m)),
                    "target_overlap_rate": float(np.mean(overlaps)),
                    "ret_corr": float(m["net_ret_a"].corr(m["net_ret_b"])) if len(m) >= 3 else np.nan,
                    "cum_ret_a_common": float((1.0 + m["net_ret_a"].astype(float)).prod() - 1.0),
                    "cum_ret_b_common": float((1.0 + m["net_ret_b"].astype(float)).prod() - 1.0),
                })
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["top_n", "common_periods", "target_overlap_rate"], ascending=[True, False, False])
    out.to_csv(V11_STABILITY_CSV, index=False)
    return out


v11_cost_weekly_df = build_v11_cost_weekly(weekly_df, score_df)
v11_cost_summary_df = build_v11_cost_summary(v11_cost_weekly_df)
v11_common_oos_df = build_v11_common_oos(v11_cost_weekly_df)
v11_cost_stability_df = build_v11_cost_stability(v11_cost_weekly_df)

print("V11 cost weekly:", v11_cost_weekly_df.shape)
print("V11 cost summary:", v11_cost_summary_df.shape)
print("V11 common OOS:", v11_common_oos_df.shape)
print("V11 cost stability:", v11_cost_stability_df.shape)
display(v11_cost_summary_df.head(20))

In [ ]:
# =========================
# 8. V11 report export
# =========================
def fmt_pct(x):
    if pd.isnull(x):
        return "nan"
    return "%.2f%%" % (100.0 * float(x))


def build_v11_report():
    lines = []
    lines.append("# ETF V11 Validation Report")
    lines.append("")
    lines.append("## Configuration")
    lines.append("")
    lines.append("- Data start: `%s`" % START_DATE)
    lines.append("- Data end: `%s`" % END_DATE)
    lines.append("- Cost rate: `%.6f` per traded notional, 万分之一 / 1 bp" % COST_RATE)
    lines.append("- Train windows: `%s`" % str(TRAIN_WINDOWS))
    lines.append("- Top N: `%s`" % str(TOP_N_LIST))
    lines.append("")
    lines.append("## Data")
    lines.append("")
    if "panel" in globals() and panel is not None and not panel.empty:
        lines.append("- Panel rows: `%d`" % len(panel))
        lines.append("- Weeks: `%d`" % panel["feature_date"].nunique())
        lines.append("- Feature date range: `%s` to `%s`" % (pd.to_datetime(panel["feature_date"]).min().date(), pd.to_datetime(panel["feature_date"]).max().date()))
    if "manifest_df" in globals() and manifest_df is not None and not manifest_df.empty:
        lines.append("- Models trained: `%d`" % len(manifest_df))
    lines.append("")
    lines.append("## Best Cost-Aware Rows")
    lines.append("")
    if "v11_cost_summary_df" in globals() and v11_cost_summary_df is not None and not v11_cost_summary_df.empty:
        cols = ["model_name", "top_n", "net_cum_ret", "net_max_drawdown", "net_win_rate", "avg_turnover_traded_notional", "net_excess_vs_random", "net_excess_vs_median"]
        for _, r in v11_cost_summary_df.sort_values("net_cum_ret", ascending=False).head(10)[cols].iterrows():
            lines.append("- `%s` top%d: net %s, mdd %s, win %s, turnover %.3f, excess random %s, excess median %s" % (
                r["model_name"], int(r["top_n"]), fmt_pct(r["net_cum_ret"]), fmt_pct(r["net_max_drawdown"]), fmt_pct(r["net_win_rate"]),
                float(r["avg_turnover_traded_notional"]), fmt_pct(r["net_excess_vs_random"]), fmt_pct(r["net_excess_vs_median"])
            ))
    else:
        lines.append("- No cost-aware summary generated.")
    lines.append("")
    lines.append("## Common OOS")
    lines.append("")
    if "v11_common_oos_df" in globals() and v11_common_oos_df is not None and not v11_common_oos_df.empty:
        cols = ["common_oos_start", "model_name", "top_n", "common_weeks", "net_cum_ret", "net_max_drawdown", "random_cum_ret", "median_cum_ret"]
        for _, r in v11_common_oos_df.head(20)[cols].iterrows():
            lines.append("- `%s` `%s` top%d weeks=%d: net %s, mdd %s, random %s, median %s" % (
                r["common_oos_start"], r["model_name"], int(r["top_n"]), int(r["common_weeks"]),
                fmt_pct(r["net_cum_ret"]), fmt_pct(r["net_max_drawdown"]), fmt_pct(r["random_cum_ret"]), fmt_pct(r["median_cum_ret"])
            ))
    else:
        lines.append("- No common OOS rows generated.")
    lines.append("")
    lines.append("## Stability")
    lines.append("")
    if "v11_cost_stability_df" in globals() and v11_cost_stability_df is not None and not v11_cost_stability_df.empty:
        for _, r in v11_cost_stability_df.head(12).iterrows():
            lines.append("- top%d `%s` vs `%s`: periods=%d, overlap=%.3f, ret_corr=%s" % (
                int(r["top_n"]), r["model_a"], r["model_b"], int(r["common_periods"]), float(r["target_overlap_rate"]),
                "nan" if pd.isnull(r["ret_corr"]) else "%.3f" % float(r["ret_corr"])
            ))
    else:
        lines.append("- No stability rows generated.")
    lines.append("")
    lines.append("## Next Experiment Recommendation")
    lines.append("")
    lines.append("- If top3 beats median/random in common OOS after cost, test portfolio rules next: top5, score-gap filter, weak-signal cash, and training-window ensemble.")
    lines.append("- If common OOS is weak or unstable, do not add features yet; first inspect drawdown events and ETF theme concentration.")
    lines.append("- If names/groups are missing, fix ETF metadata before making theme-cap conclusions.")
    text = "\\n".join(lines) + "\\n"
    with open(V11_REPORT_MD, "w", encoding="utf-8") as f:
        f.write(text)
    return text


v11_report_text = build_v11_report()
print("V11 report:", V11_REPORT_MD)
print(v11_report_text[:2000])
print("V11 outputs:")
for p in [V11_COST_WEEKLY_CSV, V11_COST_SUMMARY_CSV, V11_COMMON_OOS_CSV, V11_STABILITY_CSV, V11_REPORT_MD]:
    print(" -", p)

In [ ]:
# =========================
# 6. Latest targets for quick inspection
# =========================
latest_rows = []
if not score_df.empty:
    for model_name, gdf0 in score_df.groupby("model_name"):
        latest_date = pd.to_datetime(gdf0["feature_date"]).max()
        gdf = gdf0[pd.to_datetime(gdf0["feature_date"]) == latest_date].copy()
        for top_n in TOP_N_LIST:
            for group_cap in GROUP_CAP_LIST:
                one = select_targets(gdf, min(top_n, len(gdf)), group_cap)
                for rank_idx, (_, row) in enumerate(one.iterrows(), 1):
                    latest_rows.append({
                        "model_name": model_name,
                        "model_file": row.get("model_file", ""),
                        "target_name": row.get("target_name", ""),
                        "prediction_horizon_days": row.get("prediction_horizon_days", np.nan),
                        "rebalance_mode": row.get("rebalance_mode", ""),
                        "rebalance_interval_weeks": row.get("rebalance_interval_weeks", np.nan),
                        "train_tag": row.get("train_tag", ""),
                        "min_avg_money_20": row.get("min_avg_money_20", np.nan),
                        "money_threshold_tag": row.get("money_threshold_tag", ""),
                        "feature_date": latest_date,
                        "top_n": int(top_n),
                        "group_cap": int(group_cap),
                        "rank": int(rank_idx),
                        "code": row.get("code", ""),
                        "name": row.get("name", ""),
                        "etf_group": row.get("etf_group", ""),
                        "score": row.get("score", np.nan),
                        "eval_ret_col": row.get("eval_ret_col", ""),
                        "eval_ret": row.get("eval_ret", np.nan),
                    })
latest_targets_df = pd.DataFrame(latest_rows)
latest_targets_df.to_csv(LATEST_TARGETS_CSV, index=False)
print("latest targets:", latest_targets_df.shape)
display(latest_targets_df.head(80))

print("outputs saved:")
for p in [PANEL_CSV, SCORE_CSV, MODEL_MANIFEST_CSV, WEEKLY_PROXY_CSV, SUMMARY_CSV, YEARLY_CSV, MONEY_THRESHOLD_SUMMARY_CSV, CANDIDATE_MODELS_CSV, TARGET_OVERLAP_CSV, SCORE_GAP_DIAGNOSTICS_CSV, DRAWDOWN_EVENTS_CSV, LATEST_TARGETS_CSV]:
    print(" -", p)
print("models dir:", MODEL_DIR)
print("candidate models dir:", CANDIDATE_MODEL_DIR)

## Self Review

- V11 只验证训练窗口稳定性：固定 `ret5d`、`money5000w`、V2/V9 特征、LGB、周调仓、无主题 cap、无风控。
- 训练窗口分为固定 2021 起点和较近 2023/2024 起点，避免把起点和终点混在一个结论里。
- Notebook 保留 `REBUILD_DATA=AUTO`：有缓存优先读缓存；缺少缓存时在聚宽环境重建。
- 长循环保留 `tqdm`/fallback 进度条；聚宽 API 不用 `globals()` 检查。
- 输出包含 target overlap、score gap、drawdown events，用于判断稳定性和失效路径。
- 这版不新增特征；特征增强留给 V11，避免变量混杂。